# Joint + Depth 3D Interactive Viewer

This notebook reads a `capture_depth_joints.py` output directory and turns the saved pose keypoints plus metric depth estimates into camera-referenced 3D skeleton plots. The camera is treated as a fixed origin, with `X` moving right in the image, `Y` moving upward, and `Z` moving away from the camera.

The notebook also includes an assumed human height variable and standard body-proportion constants so measured joint distances can be compared against expected proportions or used to normalize the plotted skeleton size.

## Dependency Note

This uses Plotly for interactive 3D figures, matching the style of `depth_anything_metric_depth_interactive.ipynb`. If Plotly is missing in the active kernel, run this once and restart the kernel:

```python
%pip install plotly
```

In [ ]:
from pathlib import Path

# The notebook is intended to live at the project root. The fallback keeps it
# working if Jupyter starts in a different current working directory.
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "capture_depth_joints.py").exists():
    PROJECT_ROOT = Path(r"C:\Users\jjaramil\OneDrive - InterSystems Corporation\Documents\Muay-ThAI")

DEFAULT_RUN_DIR = (
    PROJECT_ROOT
    / "output"
    / "rodtang-taetat-2__yolov8n-pose-depth-anything-v2-metric-hypersim-vitl__20260707-155638"
)

# Point this at any capture_depth_joints.py output directory.
RUN_DIR = DEFAULT_RUN_DIR

# Anthropometric assumption used for expected joint distances.
ASSUMED_PERSON_HEIGHT_M = 1.68

# Depth-to-3D camera model. If you know your camera's FOV, put it here.
HORIZONTAL_FOV_DEG = 70.0

# Pose/person controls.
PERSON_INDEX = 0
MIN_KEYPOINT_CONFIDENCE = 0.25
SELECTED_RECORD_INDEX = 0

# If True, the notebook reloads raw depth files and resamples depth at each
# keypoint. Leave False to use the depth_m values saved in JSONL.
REFRESH_KEYPOINT_DEPTH_FROM_FILES = False
KEYPOINT_DEPTH_RADIUS_PX = 2

# Plot controls. Scaling keeps the pelvis at its camera-space position and
# scales body offsets around it according to standard proportions. Constraint
# mode projects the skeleton onto expected bone lengths while keeping the body
# anchored in camera space.
NORMALIZE_BODY_TO_ASSUMED_HEIGHT = False
CONSTRAIN_TO_BODY_PROPORTIONS = True
PROPORTION_CONSTRAINT_BLEND = 1.0  # 0.0 = raw joints, 1.0 = fully constrained
PROPORTION_CONSTRAINT_ITERATIONS = 30
SLIDER_FRAME_STRIDE = 1
MAX_SLIDER_FRAMES = 80

if not RUN_DIR.exists():
    candidates = sorted(
        (PROJECT_ROOT / "output").glob("*__yolov8n-pose-depth-anything-v2-*"),
        key=lambda path: path.stat().st_mtime,
        reverse=True,
    )
    if not candidates:
        raise FileNotFoundError(f"No combined depth/joints output found under {PROJECT_ROOT / 'output'}")
    RUN_DIR = candidates[0]

print(f"Project root: {PROJECT_ROOT}")
print(f"Using run directory: {RUN_DIR}")

In [ ]:
import json
import math
from functools import lru_cache

import numpy as np
import plotly.graph_objects as go


def resolve_run_path(value):
    if value in (None, ""):
        return None
    candidate = Path(value)
    if candidate.is_absolute():
        return candidate
    return RUN_DIR / candidate


def find_metadata_path(run_dir):
    metadata_files = sorted(run_dir.glob("*_metadata.json"))
    if not metadata_files:
        raise FileNotFoundError(f"No *_metadata.json found in {run_dir}")
    return metadata_files[0]


def find_predictions_path(run_dir, metadata):
    outputs = metadata.get("outputs", {})
    for key in ("predictions", "frame_predictions"):
        path = resolve_run_path(outputs.get(key) or metadata.get(key))
        if path is not None and path.exists():
            return path
    candidates = sorted(run_dir.glob("*_predictions.jsonl")) + sorted(run_dir.glob("*_predictions.json"))
    if not candidates:
        raise FileNotFoundError(f"No predictions JSON or JSONL found in {run_dir}")
    return candidates[0]


def load_prediction_records(path):
    if path.suffix.lower() == ".jsonl":
        return [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]
    return [json.loads(path.read_text(encoding="utf-8"))]


@lru_cache(maxsize=32)
def load_depth_array(depth_file):
    path = resolve_run_path(depth_file)
    if path is None or not path.exists():
        raise FileNotFoundError(f"Depth file not found: {depth_file}")
    data = np.load(path)
    if "depth" in data:
        return np.asarray(data["depth"], dtype=np.float32)
    return np.asarray(data[data.files[0]], dtype=np.float32)


def sample_depth_from_record(record, x, y, radius=2):
    depth_file = record.get("depth", {}).get("file")
    if not depth_file:
        return None
    depth = load_depth_array(depth_file)
    image_width = int(record.get("image_width", depth.shape[1]))
    image_height = int(record.get("image_height", depth.shape[0]))
    depth_x = int(round(float(x) * (depth.shape[1] - 1) / max(image_width - 1, 1)))
    depth_y = int(round(float(y) * (depth.shape[0] - 1) / max(image_height - 1, 1)))
    if depth_x < 0 or depth_y < 0 or depth_x >= depth.shape[1] or depth_y >= depth.shape[0]:
        return None
    radius = max(0, int(radius))
    x0 = max(0, depth_x - radius)
    x1 = min(depth.shape[1], depth_x + radius + 1)
    y0 = max(0, depth_y - radius)
    y1 = min(depth.shape[0], depth_y + radius + 1)
    finite = depth[y0:y1, x0:x1][np.isfinite(depth[y0:y1, x0:x1])]
    if finite.size == 0:
        return None
    return float(np.median(finite))


metadata_path = find_metadata_path(RUN_DIR)
metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
predictions_path = find_predictions_path(RUN_DIR, metadata)
records = load_prediction_records(predictions_path)

if not records:
    raise ValueError(f"No prediction records found in {predictions_path}")

raw_depth_count = sum(1 for record in records if record.get("depth", {}).get("file"))
print(f"Loaded {len(records)} prediction records")
print(f"Metadata: {metadata_path.name}")
print(f"Predictions: {predictions_path.name}")
print(f"Records with raw depth files: {raw_depth_count}/{len(records)}")

## 3D Reconstruction Model

Each keypoint is converted from image coordinates plus `depth_m` into a camera-centered point using a simple pinhole camera model. Without calibrated intrinsics, `HORIZONTAL_FOV_DEG` controls the lateral and vertical spread. The metric `Z` coordinate comes from Depth Anything's depth estimate.

The body-proportion table below is intentionally editable. It is used to compute expected segment lengths from `ASSUMED_PERSON_HEIGHT_M`, optionally normalize the plotted skeleton around the pelvis, and optionally constrain connected joint pairs to anatomically expected lengths. The constrained skeleton is best read as a cleaned visualization layer; it does not change the saved model outputs.

In [ ]:
KEYPOINT_NAMES = [
    "nose",
    "left_eye",
    "right_eye",
    "left_ear",
    "right_ear",
    "left_shoulder",
    "right_shoulder",
    "left_elbow",
    "right_elbow",
    "left_wrist",
    "right_wrist",
    "left_hip",
    "right_hip",
    "left_knee",
    "right_knee",
    "left_ankle",
    "right_ankle",
]

SKELETON_EDGES = [
    ("left_shoulder", "right_shoulder"),
    ("left_shoulder", "left_elbow"),
    ("left_elbow", "left_wrist"),
    ("right_shoulder", "right_elbow"),
    ("right_elbow", "right_wrist"),
    ("left_shoulder", "left_hip"),
    ("right_shoulder", "right_hip"),
    ("left_hip", "right_hip"),
    ("left_hip", "left_knee"),
    ("left_knee", "left_ankle"),
    ("right_hip", "right_knee"),
    ("right_knee", "right_ankle"),
    ("nose", "left_eye"),
    ("nose", "right_eye"),
    ("left_eye", "left_ear"),
    ("right_eye", "right_ear"),
]

# Approximate adult body segment lengths as fractions of standing height.
# These are useful sanity checks, not medical or biometric measurements.
STANDARD_BONE_FRACTIONS = {
    ("left_shoulder", "right_shoulder"): 0.259,
    ("left_hip", "right_hip"): 0.191,
    ("left_shoulder", "left_elbow"): 0.186,
    ("right_shoulder", "right_elbow"): 0.186,
    ("left_elbow", "left_wrist"): 0.146,
    ("right_elbow", "right_wrist"): 0.146,
    ("left_shoulder", "left_hip"): 0.288,
    ("right_shoulder", "right_hip"): 0.288,
    ("left_hip", "left_knee"): 0.245,
    ("right_hip", "right_knee"): 0.245,
    ("left_knee", "left_ankle"): 0.246,
    ("right_knee", "right_ankle"): 0.246,
}

TRACKED_BONES = [
    ("left_shoulder", "left_elbow"),
    ("left_elbow", "left_wrist"),
    ("right_shoulder", "right_elbow"),
    ("right_elbow", "right_wrist"),
    ("left_hip", "left_knee"),
    ("left_knee", "left_ankle"),
    ("right_hip", "right_knee"),
    ("right_knee", "right_ankle"),
]


def expected_length_m(joint_a, joint_b):
    fraction = STANDARD_BONE_FRACTIONS.get((joint_a, joint_b))
    if fraction is None:
        fraction = STANDARD_BONE_FRACTIONS.get((joint_b, joint_a))
    if fraction is None:
        return None
    return float(ASSUMED_PERSON_HEIGHT_M * fraction)


def camera_intrinsics(width, height, horizontal_fov_deg):
    horizontal_fov_rad = math.radians(float(horizontal_fov_deg))
    fx = (float(width) / 2.0) / math.tan(horizontal_fov_rad / 2.0)
    fy = fx
    cx = (float(width) - 1.0) / 2.0
    cy = (float(height) - 1.0) / 2.0
    return fx, fy, cx, cy


def pixel_depth_to_camera_xyz(x, y, depth_m, width, height):
    fx, fy, cx, cy = camera_intrinsics(width, height, HORIZONTAL_FOV_DEG)
    z = float(depth_m)
    camera_x = (float(x) - cx) * z / fx
    camera_y = -(float(y) - cy) * z / fy
    return np.array([camera_x, camera_y, z], dtype=np.float64)


def get_person(record, person_index=PERSON_INDEX):
    people = record.get("joints", {}).get("people", [])
    if not people:
        return None
    person_index = int(person_index)
    if person_index < len(people):
        return people[person_index]
    return people[0]


def record_camera_points(record, person_index=PERSON_INDEX):
    person = get_person(record, person_index)
    if person is None:
        return {}, {}
    width = int(record.get("image_width", 0))
    height = int(record.get("image_height", 0))
    points = {}
    keypoint_meta = {}
    for keypoint in person.get("keypoints", []):
        name = keypoint.get("name")
        confidence = keypoint.get("confidence")
        if confidence is not None and confidence < MIN_KEYPOINT_CONFIDENCE:
            continue
        depth_m = keypoint.get("depth_m")
        if REFRESH_KEYPOINT_DEPTH_FROM_FILES or depth_m is None:
            sampled_depth = sample_depth_from_record(
                record,
                keypoint.get("x"),
                keypoint.get("y"),
                radius=KEYPOINT_DEPTH_RADIUS_PX,
            )
            if sampled_depth is not None:
                depth_m = sampled_depth
        if depth_m is None or not np.isfinite(depth_m):
            continue
        points[name] = pixel_depth_to_camera_xyz(
            keypoint.get("x"),
            keypoint.get("y"),
            depth_m,
            width,
            height,
        )
        keypoint_meta[name] = {**keypoint, "depth_m": float(depth_m)}
    return points, keypoint_meta


def point_distance(points, joint_a, joint_b):
    if joint_a not in points or joint_b not in points:
        return None
    return float(np.linalg.norm(points[joint_a] - points[joint_b]))


def midpoint(points, joint_a, joint_b):
    if joint_a in points and joint_b in points:
        return 0.5 * (points[joint_a] + points[joint_b])
    return None


def body_anchor(points):
    pelvis = midpoint(points, "left_hip", "right_hip")
    if pelvis is not None:
        return pelvis
    shoulders = midpoint(points, "left_shoulder", "right_shoulder")
    if shoulders is not None:
        return shoulders
    if points:
        return np.mean(np.vstack(list(points.values())), axis=0)
    return np.zeros(3, dtype=np.float64)


def proportion_scale_factor(points):
    ratios = []
    for (joint_a, joint_b), fraction in STANDARD_BONE_FRACTIONS.items():
        observed = point_distance(points, joint_a, joint_b)
        if observed is None or observed <= 1e-6:
            continue
        expected = ASSUMED_PERSON_HEIGHT_M * fraction
        ratios.append(expected / observed)
    if not ratios:
        return 1.0
    return float(np.median(ratios))


def constrained_body_points(points):
    if not points:
        return points

    constrained = {name: point.copy() for name, point in points.items()}
    constraints = []
    for joint_a, joint_b in STANDARD_BONE_FRACTIONS:
        if joint_a in constrained and joint_b in constrained:
            constraints.append((joint_a, joint_b, expected_length_m(joint_a, joint_b)))

    if not constraints:
        return constrained

    anchor = body_anchor(constrained)
    iterations = max(1, int(PROPORTION_CONSTRAINT_ITERATIONS))
    for _ in range(iterations):
        for joint_a, joint_b, target_length in constraints:
            if target_length is None or target_length <= 0:
                continue
            delta = constrained[joint_b] - constrained[joint_a]
            distance = float(np.linalg.norm(delta))
            if distance <= 1e-8:
                continue
            midpoint_value = 0.5 * (constrained[joint_a] + constrained[joint_b])
            target_delta = delta * (target_length / distance)
            constrained[joint_a] = midpoint_value - 0.5 * target_delta
            constrained[joint_b] = midpoint_value + 0.5 * target_delta

        # Keep the fitted skeleton attached to the same camera-space body anchor.
        anchor_shift = anchor - body_anchor(constrained)
        for name in constrained:
            constrained[name] = constrained[name] + anchor_shift

    blend = float(np.clip(PROPORTION_CONSTRAINT_BLEND, 0.0, 1.0))
    if blend < 1.0:
        constrained = {
            name: points[name] * (1.0 - blend) + constrained[name] * blend
            for name in constrained
        }

    return constrained


def scaled_body_points(
    points,
    normalize=NORMALIZE_BODY_TO_ASSUMED_HEIGHT,
    constrain=CONSTRAIN_TO_BODY_PROPORTIONS,
):
    if not points:
        return points, 1.0
    anchor = body_anchor(points)
    scale = proportion_scale_factor(points) if normalize else 1.0
    scaled = {name: anchor + (point - anchor) * scale for name, point in points.items()}
    if constrain:
        scaled = constrained_body_points(scaled)
    return scaled, scale


example_points, _ = record_camera_points(records[SELECTED_RECORD_INDEX])
print(f"Usable keypoints in selected record: {len(example_points)}")
print(f"Anthropometric scale factor for selected record: {proportion_scale_factor(example_points):.3f}")

## Plot Helpers

These helpers build Plotly traces from any processed frame. Hover over joints to inspect camera coordinates, depth, confidence, and source pixel locations.

In [ ]:
def ordered_point_names(points):
    ordered = [name for name in KEYPOINT_NAMES if name in points]
    ordered.extend(sorted(name for name in points if name not in ordered))
    return ordered


def skeleton_line_xyz(points):
    xs, ys, zs = [], [], []
    for joint_a, joint_b in SKELETON_EDGES:
        if joint_a in points and joint_b in points:
            xs.extend([points[joint_a][0], points[joint_b][0], None])
            ys.extend([points[joint_a][1], points[joint_b][1], None])
            zs.extend([points[joint_a][2], points[joint_b][2], None])
    return xs, ys, zs


def joint_hover_text(names, points, keypoint_meta, scale_factor, constrain):
    hovers = []
    for name in names:
        point = points[name]
        meta = keypoint_meta.get(name, {})
        confidence = meta.get("confidence")
        confidence_text = "n/a" if confidence is None else f"{confidence:.3f}"
        hovers.append(
            "<br>".join(
                [
                    f"joint: {name}",
                    f"X right: {point[0]:.3f} m",
                    f"Y up: {point[1]:.3f} m",
                    f"Z depth: {point[2]:.3f} m",
                    f"pixel: ({meta.get('x', float('nan')):.1f}, {meta.get('y', float('nan')):.1f})",
                    f"confidence: {confidence_text}",
                    f"body scale: {scale_factor:.3f}",
                    f"constraint: {'on' if constrain else 'off'}",
                ]
            )
        )
    return hovers


def skeleton_traces(
    record_index,
    person_index=PERSON_INDEX,
    normalize=NORMALIZE_BODY_TO_ASSUMED_HEIGHT,
    constrain=CONSTRAIN_TO_BODY_PROPORTIONS,
):
    record = records[int(record_index)]
    raw_points, keypoint_meta = record_camera_points(record, person_index)
    points, scale_factor = scaled_body_points(raw_points, normalize=normalize, constrain=constrain)
    names = ordered_point_names(points)
    line_x, line_y, line_z = skeleton_line_xyz(points)
    marker_points = np.vstack([points[name] for name in names]) if names else np.empty((0, 3))
    frame_label = record.get("source_frame_index", record.get("frame_index", record_index))
    mode_label = "constrained" if constrain else "raw"
    title = f"record {record_index} | source frame {frame_label} | {mode_label} | scale {scale_factor:.3f}"
    return [
        go.Scatter3d(
            x=line_x,
            y=line_y,
            z=line_z,
            mode="lines",
            line={"width": 7, "color": "#13a37f"},
            name="skeleton",
            hoverinfo="skip",
        ),
        go.Scatter3d(
            x=marker_points[:, 0] if len(marker_points) else [],
            y=marker_points[:, 1] if len(marker_points) else [],
            z=marker_points[:, 2] if len(marker_points) else [],
            mode="markers+text",
            text=names,
            textposition="top center",
            marker={"size": 5, "color": "#1f77b4"},
            name="joints",
            customdata=joint_hover_text(names, points, keypoint_meta, scale_factor, constrain),
            hovertemplate="%{customdata}<extra></extra>",
        ),
        go.Scatter3d(
            x=[0.0],
            y=[0.0],
            z=[0.0],
            mode="markers+text",
            text=["camera"],
            textposition="bottom center",
            marker={"size": 6, "color": "black", "symbol": "diamond"},
            name="camera origin",
            hovertemplate="camera origin<br>X: 0 m<br>Y: 0 m<br>Z: 0 m<extra></extra>",
        ),
    ], title, points


def axis_ranges(point_sets, pad_fraction=0.15):
    arrays = []
    for points in point_sets:
        if points:
            arrays.append(np.vstack(list(points.values())))
    arrays.append(np.zeros((1, 3), dtype=np.float64))
    all_points = np.vstack(arrays)
    mins = all_points.min(axis=0)
    maxs = all_points.max(axis=0)
    span = np.maximum(maxs - mins, 0.5)
    pad = span * pad_fraction
    return [[mins[i] - pad[i], maxs[i] + pad[i]] for i in range(3)]


def update_scene(fig, ranges=None, title=None):
    scene = {
        "xaxis_title": "X right (m)",
        "yaxis_title": "Y up (m)",
        "zaxis_title": "Z from camera (m)",
        "aspectmode": "data",
    }
    if ranges is not None:
        scene["xaxis"] = {"range": ranges[0]}
        scene["yaxis"] = {"range": ranges[1]}
        scene["zaxis"] = {"range": ranges[2]}
    fig.update_layout(
        title=title,
        width=950,
        height=750,
        margin={"l": 0, "r": 0, "t": 50, "b": 0},
        scene=scene,
        legend={"orientation": "h", "yanchor": "bottom", "y": 1.02, "xanchor": "left", "x": 0},
    )


def plot_skeleton_3d(
    record_index=SELECTED_RECORD_INDEX,
    person_index=PERSON_INDEX,
    normalize=NORMALIZE_BODY_TO_ASSUMED_HEIGHT,
    constrain=CONSTRAIN_TO_BODY_PROPORTIONS,
):
    traces, title, points = skeleton_traces(record_index, person_index, normalize=normalize, constrain=constrain)
    fig = go.Figure(data=traces)
    update_scene(fig, ranges=axis_ranges([points]), title=title)
    return fig

## Single Frame 3D Skeleton

Change `SELECTED_RECORD_INDEX`, `PERSON_INDEX`, `NORMALIZE_BODY_TO_ASSUMED_HEIGHT`, or `CONSTRAIN_TO_BODY_PROPORTIONS` in the config cell and rerun from there to inspect a different pose.

In [ ]:
fig_skeleton = plot_skeleton_3d(
    record_index=SELECTED_RECORD_INDEX,
    person_index=PERSON_INDEX,
    normalize=NORMALIZE_BODY_TO_ASSUMED_HEIGHT,
    constrain=CONSTRAIN_TO_BODY_PROPORTIONS,
)
fig_skeleton.show()

## Raw vs Constrained Overlay

This overlays the depth-derived skeleton with the proportion-constrained skeleton. Large gaps between the two are the places where the correction is doing real work, usually because of depth noise, occlusion, or a shaky keypoint.

In [ ]:
def plot_raw_vs_constrained(record_index=SELECTED_RECORD_INDEX, person_index=PERSON_INDEX, normalize=NORMALIZE_BODY_TO_ASSUMED_HEIGHT):
    raw_traces, _, raw_points = skeleton_traces(record_index, person_index, normalize=normalize, constrain=False)
    constrained_traces, title, constrained_points = skeleton_traces(record_index, person_index, normalize=normalize, constrain=True)

    raw_traces[0].update(line={"width": 4, "color": "rgba(110, 110, 110, 0.55)"}, name="raw skeleton")
    raw_traces[1].update(mode="markers", marker={"size": 4, "color": "rgba(110, 110, 110, 0.55)"}, name="raw joints")
    constrained_traces[0].update(line={"width": 8, "color": "#13a37f"}, name="constrained skeleton")
    constrained_traces[1].update(marker={"size": 5, "color": "#1f77b4"}, name="constrained joints")

    fig = go.Figure(
        data=[
            raw_traces[0],
            constrained_traces[0],
            raw_traces[1],
            constrained_traces[1],
            constrained_traces[2],
        ]
    )
    update_scene(
        fig,
        ranges=axis_ranges([raw_points, constrained_points]),
        title=f"Raw vs constrained | {title}",
    )
    return fig


fig_compare = plot_raw_vs_constrained(
    record_index=SELECTED_RECORD_INDEX,
    person_index=PERSON_INDEX,
    normalize=NORMALIZE_BODY_TO_ASSUMED_HEIGHT,
)
fig_compare.show()

## Joint Distance Checks

This table compares reconstructed 3D joint distances with expected segment lengths from `ASSUMED_PERSON_HEIGHT_M` and `STANDARD_BONE_FRACTIONS`. Ratios far from 1.0 are useful clues about occlusion, pose-estimation drift, depth noise, or a camera FOV assumption that needs tuning.

In [ ]:
def bone_length_rows(
    record_index=SELECTED_RECORD_INDEX,
    person_index=PERSON_INDEX,
    normalize=NORMALIZE_BODY_TO_ASSUMED_HEIGHT,
    constrain=CONSTRAIN_TO_BODY_PROPORTIONS,
):
    raw_points, _ = record_camera_points(records[int(record_index)], person_index)
    points, scale_factor = scaled_body_points(raw_points, normalize=normalize, constrain=constrain)
    rows = []
    for joint_a, joint_b in STANDARD_BONE_FRACTIONS:
        observed = point_distance(points, joint_a, joint_b)
        expected = expected_length_m(joint_a, joint_b)
        if observed is None or expected is None:
            continue
        rows.append(
            {
                "segment": f"{joint_a} to {joint_b}",
                "observed_m": observed,
                "expected_m": expected,
                "observed_expected_ratio": observed / expected if expected > 0 else np.nan,
                "scale_factor": scale_factor,
            }
        )
    return rows


def plot_bone_length_table(
    record_index=SELECTED_RECORD_INDEX,
    person_index=PERSON_INDEX,
    normalize=NORMALIZE_BODY_TO_ASSUMED_HEIGHT,
    constrain=CONSTRAIN_TO_BODY_PROPORTIONS,
):
    rows = bone_length_rows(record_index, person_index, normalize=normalize, constrain=constrain)
    if not rows:
        raise ValueError("No standard bone segments were available in this frame.")
    fig = go.Figure(
        data=[
            go.Table(
                header={"values": ["segment", "observed m", "expected m", "observed / expected"], "fill_color": "#f0f3f6", "align": "left"},
                cells={
                    "values": [
                        [row["segment"] for row in rows],
                        [f"{row['observed_m']:.3f}" for row in rows],
                        [f"{row['expected_m']:.3f}" for row in rows],
                        [f"{row['observed_expected_ratio']:.2f}" for row in rows],
                    ],
                    "align": "left",
                },
            )
        ]
    )
    mode_label = "constrained" if constrain else "raw"
    fig.update_layout(title=f"Joint distance checks for record {record_index} ({mode_label})", width=900, height=520)
    return fig


fig_distances = plot_bone_length_table(
    record_index=SELECTED_RECORD_INDEX,
    person_index=PERSON_INDEX,
    normalize=NORMALIZE_BODY_TO_ASSUMED_HEIGHT,
    constrain=CONSTRAIN_TO_BODY_PROPORTIONS,
)
fig_distances.show()

## 3D Skeleton Slider

The slider animates the camera-referenced skeleton across processed frames. For long videos, increase `SLIDER_FRAME_STRIDE` or lower `MAX_SLIDER_FRAMES` in the config cell to keep the notebook responsive.

In [ ]:
def slider_record_indices():
    stride = max(1, int(SLIDER_FRAME_STRIDE))
    indices = list(range(0, len(records), stride))
    if MAX_SLIDER_FRAMES is not None:
        indices = indices[: int(MAX_SLIDER_FRAMES)]
    return indices


def plot_skeleton_slider(
    person_index=PERSON_INDEX,
    normalize=NORMALIZE_BODY_TO_ASSUMED_HEIGHT,
    constrain=CONSTRAIN_TO_BODY_PROPORTIONS,
):
    indices = slider_record_indices()
    if not indices:
        raise ValueError("No records selected for the slider.")
    prepared = []
    for record_index in indices:
        traces, title, points = skeleton_traces(record_index, person_index, normalize=normalize, constrain=constrain)
        prepared.append((record_index, traces, title, points))
    ranges = axis_ranges([item[3] for item in prepared])
    fig = go.Figure(data=prepared[0][1])
    fig.frames = [
        go.Frame(
            name=str(record_index),
            data=traces,
            layout=go.Layout(title=title),
        )
        for record_index, traces, title, _ in prepared
    ]
    steps = [
        {
            "method": "animate",
            "label": str(record_index),
            "args": [
                [str(record_index)],
                {"mode": "immediate", "frame": {"duration": 0, "redraw": True}, "transition": {"duration": 0}},
            ],
        }
        for record_index, _, _, _ in prepared
    ]
    fig.update_layout(
        sliders=[
            {
                "active": 0,
                "currentvalue": {"prefix": "record "},
                "pad": {"t": 40},
                "steps": steps,
            }
        ],
        updatemenus=[
            {
                "type": "buttons",
                "direction": "left",
                "x": 0,
                "y": 0,
                "showactive": False,
                "buttons": [
                    {
                        "label": "Play",
                        "method": "animate",
                        "args": [None, {"frame": {"duration": 120, "redraw": True}, "fromcurrent": True}],
                    },
                    {
                        "label": "Pause",
                        "method": "animate",
                        "args": [[None], {"frame": {"duration": 0, "redraw": False}, "mode": "immediate"}],
                    },
                ],
            }
        ],
    )
    update_scene(fig, ranges=ranges, title=prepared[0][2])
    return fig


fig_slider = plot_skeleton_slider(
    person_index=PERSON_INDEX,
    normalize=NORMALIZE_BODY_TO_ASSUMED_HEIGHT,
    constrain=CONSTRAIN_TO_BODY_PROPORTIONS,
)
fig_slider.show()

## Proportion Ratios Over Time

This plot tracks observed segment length divided by expected segment length. A ratio near 1.0 means the reconstructed segment agrees with the assumed-height body model for that frame.

In [ ]:
def record_time_value(record):
    if record.get("time_s") is not None:
        return float(record["time_s"])
    return float(record.get("frame_index", 0))


def plot_proportion_ratios_over_time(
    person_index=PERSON_INDEX,
    normalize=NORMALIZE_BODY_TO_ASSUMED_HEIGHT,
    constrain=CONSTRAIN_TO_BODY_PROPORTIONS,
):
    fig = go.Figure()
    for joint_a, joint_b in TRACKED_BONES:
        expected = expected_length_m(joint_a, joint_b)
        if expected is None or expected <= 0:
            continue
        xs = []
        ys = []
        hover = []
        for record_index, record in enumerate(records):
            raw_points, _ = record_camera_points(record, person_index)
            points, scale_factor = scaled_body_points(raw_points, normalize=normalize, constrain=constrain)
            observed = point_distance(points, joint_a, joint_b)
            if observed is None:
                continue
            xs.append(record_time_value(record))
            ys.append(observed / expected)
            hover.append(
                f"record {record_index}<br>{joint_a} to {joint_b}<br>observed: {observed:.3f} m<br>expected: {expected:.3f} m<br>scale: {scale_factor:.3f}"
            )
        if xs:
            fig.add_trace(
                go.Scatter(
                    x=xs,
                    y=ys,
                    mode="lines+markers",
                    name=f"{joint_a} to {joint_b}",
                    customdata=hover,
                    hovertemplate="%{customdata}<br>ratio: %{y:.2f}<extra></extra>",
                )
            )
    fig.add_hline(y=1.0, line_dash="dash", line_color="black", annotation_text="expected")
    mode_label = "constrained" if constrain else "raw"
    fig.update_layout(
        title=f"Observed joint distance / expected body proportion ({mode_label})",
        width=950,
        height=550,
        xaxis_title="time (s) or frame index",
        yaxis_title="observed / expected",
        hovermode="closest",
    )
    return fig


fig_ratios = plot_proportion_ratios_over_time(
    person_index=PERSON_INDEX,
    normalize=NORMALIZE_BODY_TO_ASSUMED_HEIGHT,
    constrain=CONSTRAIN_TO_BODY_PROPORTIONS,
)
fig_ratios.show()

## Programmatic Lookup

Use these helpers when you want exact numeric coordinates or distances for downstream analysis.

In [ ]:
def joint_xyz(
    record_index,
    joint_name,
    person_index=PERSON_INDEX,
    normalize=NORMALIZE_BODY_TO_ASSUMED_HEIGHT,
    constrain=CONSTRAIN_TO_BODY_PROPORTIONS,
):
    raw_points, _ = record_camera_points(records[int(record_index)], person_index)
    points, _ = scaled_body_points(raw_points, normalize=normalize, constrain=constrain)
    if joint_name not in points:
        raise KeyError(f"Joint {joint_name!r} is not available for record {record_index}")
    return tuple(float(value) for value in points[joint_name])


def joint_distance(
    record_index,
    joint_a,
    joint_b,
    person_index=PERSON_INDEX,
    normalize=NORMALIZE_BODY_TO_ASSUMED_HEIGHT,
    constrain=CONSTRAIN_TO_BODY_PROPORTIONS,
):
    raw_points, _ = record_camera_points(records[int(record_index)], person_index)
    points, _ = scaled_body_points(raw_points, normalize=normalize, constrain=constrain)
    distance = point_distance(points, joint_a, joint_b)
    if distance is None:
        raise KeyError(f"Could not compute distance for {joint_a!r} to {joint_b!r} in record {record_index}")
    return distance


print("Example left wrist XYZ:", joint_xyz(SELECTED_RECORD_INDEX, "left_wrist"))
print(
    "Example left upper arm distance:",
    f"{joint_distance(SELECTED_RECORD_INDEX, 'left_shoulder', 'left_elbow'):.3f} m",
)